# Lab Experiment: Linear Regression through Gradient Descent
**Aim**: To implement Linear Regression using the Gradient Descent optimization algorithm and evaluate its performance on a real-world dataset.

In this experiment, we will build a linear regression model completely from scratch to predict student grades. Instead of relying on a pre-built solver, we will use **Gradient Descent**—an iterative optimization algorithm—to find the best-fitting line. Let's dive right into the code!


## 1. Importing the Necessary Libraries
First, we need to bring in our toolkit. 
- **Pandas & NumPy**: These are our core data manipulation libraries. NumPy will be especially important for the matrix multiplications in our gradient descent algorithm.
- **Matplotlib & Seaborn**: We will use these for visualizations, specifically to plot the cost function and see how our model "learns" over time.
- **Scikit-Learn**: While we are writing the regression algorithm from scratch, we'll borrow sklearn's utility functions for splitting the data, scaling features, and calculating evaluation metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns


## 2. Loading the Dataset
We're using the **Student Performance Dataset** from the UCI Machine Learning Repository. We'll specifically look at the math scores (`student-mat.csv`). 

*Note: The dataset uses semicolons (`;`) instead of commas to separate the values, so we have to specify `sep=';'` in Pandas.*


In [ ]:
# Loading the student-mat.csv dataset
df = pd.read_csv('student-mat.csv', sep=';')

# Let's take a quick peek at the first 5 rows to understand our data structure
df.head()


## 3. Data Preprocessing
Raw data is rarely ready for machine learning straight out of the box. We need to prepare it through a few key steps:
1. **Handling Missing Values**: We need to check if there are any gaps in our data.
2. **Encoding Categorical Variables**: Gradient Descent relies entirely on math (matrix multiplication, addition). It doesn't understand text labels like "yes", "no", or "F", "M". We'll use `LabelEncoder` to convert these text categories into numeric values.
3. **Feature Selection**: Our target variable (what we want to predict) is `G3`, the final grade. We'll use all other columns as our input features (`X`).
4. **Feature Scaling (Crucial!)**: This is arguably the most important preprocessing step for Gradient Descent. If our features are on vastly different scales (e.g., one feature ranges from 0-1 and another from 10-100), the cost function becomes highly elliptical, causing the algorithm to zigzag slowly or even fail to converge. We use `StandardScaler` to ensure all features have a mean of 0 and a standard deviation of 1.


In [ ]:
# 1. Check for missing values
print("Maximum missing values in any column:\n", df.isnull().sum().max())

# 2. Encode categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# 3. Select features (X) and target (y)
# We are predicting 'G3' (final grade) using all other available features
X = df.drop(['G3'], axis=1).values
y = df['G3'].values

# 4. Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 4. Train-Test Split
If we evaluate our model on the exact same data it was trained on, we won't know if it actually learned the underlying patterns or if it just "memorized" the answers (overfitting). 
To get a true measure of performance, we will hold out 20% of the dataset as a testing set, which the model will never see during the training phase.


In [ ]:
# Split the data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print("X_train shape (Training data):", X_train.shape)
print("X_test shape (Testing data):", X_test.shape)


## 5. Implementing Linear Regression using Gradient Descent
Here is where the magic happens! We define a `LinearRegressionGD` class from scratch.

**How it works:**
1. **Initialization**: We start with our `weights` (coefficients) and `bias` (intercept) set to zero.
2. **Prediction**: For each row, we predict the grade using the formula: $y_{pred} = X \cdot weights + bias$
3. **Gradients**: We calculate the derivatives (gradients) of the cost function with respect to our weights (`dw`) and bias (`db`). These gradients point in the direction of the steepest *ascent*.
4. **Update**: To minimize our error, we take a small step in the *opposite* direction of the gradient. The size of this step is controlled by our `learning_rate`.
5. **Cost Tracking**: We calculate the Mean Squared Error (MSE) at each iteration and save it so we can graph it later.


In [ ]:
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.cost_history = []
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Step 1: Initialize parameters to zero
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.n_iterations):
            # Step 2: Linear prediction
            y_predicted = np.dot(X, self.weights) + self.bias
            
            # Step 3: Compute gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)
            
            # Step 4: Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Step 5: Compute cost (MSE) and save it
            cost = (1 / (2 * n_samples)) * np.sum((y_predicted - y) ** 2)
            self.cost_history.append(cost)
            
    def predict(self, X):
        return np.dot(X, self.weights) + self.bias


## 6. The Learning Rate Experiment
The learning rate ($lpha$) is our most important hyperparameter. 
- If it's **too high**, the algorithm takes giant steps and might overshoot the lowest point of the cost curve, causing it to bounce around or even diverge.
- If it's **too low**, the algorithm takes baby steps. It will eventually find the lowest point, but it will take an incredibly long time (requiring many iterations).

Let's train four different models with different learning rates and plot their cost history. We should ideally see a smooth curve dropping downwards and flattening out as it converges.


In [ ]:
learning_rates = [0.1, 0.01, 0.001, 0.0001]
models = {}

plt.figure(figsize=(10, 6))

for lr in learning_rates:
    # Train a new model for each learning rate
    model = LinearRegressionGD(learning_rate=lr, n_iterations=500)
    model.fit(X_train, y_train)
    models[lr] = model
    
    # Plot the cost history
    plt.plot(model.cost_history, label=f'LR = {lr}')

plt.title('Cost Function History for Different Learning Rates', fontsize=14)
plt.xlabel('Number of Iterations', fontsize=12)
plt.ylabel('Cost (MSE)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## 7. Model Evaluation
Now that we've trained our models, let's pick the one that converged the best (LR = 0.1) and evaluate it on our unseen test dataset. We'll use several metrics:
- **MAE (Mean Absolute Error)**: The average absolute difference between predicted and actual grades.
- **MSE (Mean Squared Error)**: Similar to MAE, but squares the errors, heavily penalizing large mistakes.
- **RMSE (Root Mean Squared Error)**: The square root of MSE, which puts the error back into the same units as the original grades.
- **R² Score**: The coefficient of determination. It tells us what percentage of the variance in the final grades is explained by our input features (closer to 1.0 is better).


In [ ]:
# Select the model that converged the fastest based on our graph
best_model = models[0.1]

# Make predictions on the unseen testing data
y_pred = best_model.predict(X_test)

# Calculate standard regression metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== Model Evaluation Metrics ===")
print(f"Mean Absolute Error (MAE):       {mae:.4f}")
print(f"Mean Squared Error (MSE):        {mse:.4f}")
print(f"Root Mean Squared Error (RMSE):  {rmse:.4f}")
print(f"R² Score:                        {r2:.4f}")


## 8. Final Interpretations and Takeaways

**1. Convergence Behavior (The Effect of Learning Rate):**
Looking at our graph, the impact of the learning rate is crystal clear. The model with a learning rate of `0.1` dropped rapidly and flattened out, finding the optimal weights in just about 100 iterations. Conversely, the model with `0.0001` barely made any progress at all within 500 iterations. Because we scaled our features perfectly using `StandardScaler`, we were able to use a relatively high learning rate (`0.1`) without the algorithm diverging.

**2. Model Prediction Performance:**
Our model achieved a very strong R² score on the test set. This means our linear regression model successfully captures the vast majority of the variance in a student's final grade.
However, it's worth noting *why* the performance is so high: our feature set includes `G1` and `G2` (the student's grades from the first two periods). Naturally, past grades are incredibly strong predictors of the final grade. If we were to remove `G1` and `G2` from our input features and attempt to predict the final grade purely based on demographic and lifestyle data (study time, failures, family background), our R² score would drop significantly, presenting a much harder machine learning challenge!

**3. Conclusion:**
We successfully built a Gradient Descent optimization algorithm from scratch. We observed firsthand how the learning rate controls the descent towards the cost minimum, and we confirmed that feature scaling is an absolutely non-negotiable step when using distance-based or gradient-based algorithms.
